In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
from scipy.io import loadmat
from scipy.signal import hilbert, coherence, butter, filtfilt
from scipy.stats import pearsonr
from google.colab import drive

drive.mount('/content/drive')

# CONSTANTS

FS = 256
N_CELLS = 8
EPOCH_SEC = 3.0
EPOCH_LEN = int(EPOCH_SEC * FS)  # 768 samples - must match preprocessing script

CELL_CHANNELS = {
    1: ("Cz", "F4"),
    2: ("Cz", "F4"),
    3: ("Cz", "F4"),
    4: ("Cz", "F4"),
    5: ("Cz", "F4"),
    6: ("Cz", "F4"),
    7: ("O1", "F4"),
    8: ("Fz", "F4"),
}

# 5 bands, matching the paper exactly (alpha split into lower/higher)
BANDS = {
    "Delta":       (1, 4),
    "Theta":       (4, 8),
    "LowerAlpha":  (8, 10),
    "HigherAlpha": (10, 12),
    "Beta":        (12, 30),
}

PROJECT_PATH = "/content/drive/MyDrive/ADHD_EEG_Project"
PREPROCESSED_DIR = os.path.join(PROJECT_PATH, "data", "preprocessed_paper_method")
OUTPUT_DIR = os.path.join(PROJECT_PATH, "data", "features_paper_method")
os.makedirs(OUTPUT_DIR, exist_ok=True)

GROUP_LABELS = {"FC": 0, "MC": 0, "FADHD": 1, "MADHD": 1}

MEASURE_NAMES = ["Correlation", "Coherence", "PLV", "PLI"]
BAND_NAMES = list(BANDS.keys())
N_BANDS = len(BAND_NAMES)
EXPECTED_N_FEATURES = N_CELLS * N_BANDS  # 8 * 5 = 40


# LOAD DATA

def load_group(group_name):
    file_path = os.path.join(PREPROCESSED_DIR, f"{group_name}_preprocessed_paper.mat")
    mat_data = loadmat(file_path)
    return mat_data[group_name]


FC_data    = load_group("FC")
MC_data    = load_group("MC")
FADHD_data = load_group("FADHD")
MADHD_data = load_group("MADHD")

# VALIDATION HELPERS


def get_n_cells(group_data):
    """group_data has shape (1, n_cells) -> n_cells is dim 1."""
    return group_data.shape[1]


def get_n_subjects_for_cell(group_data, cell_index):
    """
    group_data[0, cell_index] has shape (1, n_subjects)
    -> n_subjects is dim 1 (NOT dim 0, which is the MATLAB singleton row).
    """
    cell_recordings = group_data[0, cell_index]
    return cell_recordings.shape[1]


def validate_group_structure(group_data, group_name):
    """
    Confirms:
      - group_data is 2D with shape (1, N_CELLS)
      - every cell reports the same subject count
    Returns the validated n_subjects for this group.
    """
    print(f"\n[VALIDATION] Group: {group_name}")

    if group_data.ndim != 2 or group_data.shape[0] != 1:
        raise ValueError(
            f"{group_name}: expected shape (1, n_cells), got {group_data.shape}"
        )

    n_cells = get_n_cells(group_data)
    print(f"  n_cells detected: {n_cells}")
    if n_cells != N_CELLS:
        raise ValueError(
            f"{group_name}: expected {N_CELLS} cells, found {n_cells}. "
            f"Check preprocessing output or N_CELLS constant."
        )

    subject_counts = []
    for cell_index in range(n_cells):
        cell_recordings = group_data[0, cell_index]
        if cell_recordings.ndim != 2 or cell_recordings.shape[0] != 1:
            raise ValueError(
                f"{group_name} cell {cell_index+1}: expected shape (1, n_subjects), "
                f"got {cell_recordings.shape}"
            )
        n_subj_this_cell = cell_recordings.shape[1]
        subject_counts.append(n_subj_this_cell)
        print(f"  Cell {cell_index+1}: {n_subj_this_cell} subjects, "
              f"container shape {cell_recordings.shape}")

    if len(set(subject_counts)) != 1:
        raise ValueError(
            f"{group_name}: inconsistent subject counts across cells: {subject_counts}. "
            f"All cells must have the same number of subjects."
        )

    n_subjects = subject_counts[0]
    print(f"  -> Validated n_subjects for {group_name}: {n_subjects}")

    # spot-check subject 0 of cell 0 for expected (n_epochs, 768, 2) shape
    sample = group_data[0, 0][0, 0]
    sample = np.squeeze(sample)
    print(f"  Sample subject0/cell0 squeezed shape: {sample.shape} "
          f"(expect (n_epochs, {EPOCH_LEN}, 2) or ({EPOCH_LEN}, 2) for 1 epoch)")

    return n_subjects


# BAND FILTERING

def bandpass_filter(signal, low, high, fs=FS, order=4):
    nyquist = 0.5 * fs
    b, a = butter(order, [low / nyquist, high / nyquist], btype="band")

    padlen = 3 * (max(len(b), len(a)) - 1)
    if signal.shape[0] <= padlen:
        raise ValueError(
            f"Signal length {signal.shape[0]} is too short for filtfilt "
            f"(needs > {padlen} samples). Check epoch extraction upstream."
        )
    return filtfilt(b, a, signal)


# CONNECTIVITY MEASURES

def get_correlation(signal1, signal2):
    corr_value, _ = pearsonr(signal1, signal2)
    return corr_value


def get_coherence(signal1, signal2, fs=FS):
    _, coh_values = coherence(signal1, signal2, fs=fs)
    return np.mean(coh_values)


def get_phase(signal):
    analytic_signal = hilbert(signal)
    return np.angle(analytic_signal)


def get_plv(signal1, signal2):
    phase1 = get_phase(signal1)
    phase2 = get_phase(signal2)
    phase_diff = phase1 - phase2
    return np.abs(np.mean(np.exp(1j * phase_diff)))


def get_pli(signal1, signal2):
    phase1 = get_phase(signal1)
    phase2 = get_phase(signal2)
    phase_diff = phase1 - phase2
    return np.abs(np.mean(np.sign(np.sin(phase_diff))))


def compute_all_measures(channel1_signal, channel2_signal):
    return {
        "Correlation": get_correlation(channel1_signal, channel2_signal),
        "Coherence":   get_coherence(channel1_signal, channel2_signal),
        "PLV":         get_plv(channel1_signal, channel2_signal),
        "PLI":         get_pli(channel1_signal, channel2_signal),
    }


# EPOCH HANDLING

def get_subject_epochs(cell_recordings, subject_index):

    if cell_recordings.ndim != 2 or subject_index >= cell_recordings.shape[1]:
        print(f"    [WARN] subject_index {subject_index} out of bounds for "
              f"cell_recordings shape {cell_recordings.shape} - returning empty.")
        return np.empty((0, EPOCH_LEN, 2))

    epochs = cell_recordings[0, subject_index]
    epochs = np.squeeze(epochs)

    if epochs.size == 0:
        return np.empty((0, EPOCH_LEN, 2))

    # a single epoch can squeeze down to 2D (EPOCH_LEN, 2) - restore the
    # epoch dimension so downstream code can always assume 3D
    if epochs.ndim == 2:
        epochs = epochs[np.newaxis, :, :]

    if epochs.ndim != 3 or epochs.shape[1] != EPOCH_LEN or epochs.shape[2] != 2:
        print(f"    [WARN] subject_index {subject_index}: malformed epoch array "
              f"shape {epochs.shape} (expected (n_epochs, {EPOCH_LEN}, 2)) - "
              f"treating as empty.")
        return np.empty((0, EPOCH_LEN, 2))

    return epochs


def compute_band_connectivity_epoched(epochs, low, high):

    if epochs.shape[0] == 0:
        return {m: np.nan for m in MEASURE_NAMES}

    per_epoch_values = {m: [] for m in MEASURE_NAMES}

    for epoch in epochs:
        ch1_band = bandpass_filter(epoch[:, 0], low, high)
        ch2_band = bandpass_filter(epoch[:, 1], low, high)

        measures = compute_all_measures(ch1_band, ch2_band)
        for name, value in measures.items():
            per_epoch_values[name].append(value)

    return {m: np.mean(vals) for m, vals in per_epoch_values.items()}


# FEATURE EXTRACTION PER GROUP

def extract_features_for_group(group_data, group_name):
    n_subjects = validate_group_structure(group_data, group_name)
    print(f"Processing group: {group_name} ({n_subjects} subjects)")

    fused_matrices = {
        measure: np.full((n_subjects, N_CELLS * N_BANDS), np.nan)
        for measure in MEASURE_NAMES
    }

    empty_epoch_warnings = 0

    for cell_index in range(N_CELLS):
        cell_recordings = group_data[0, cell_index]

        for subject_index in range(n_subjects):
            epochs = get_subject_epochs(cell_recordings, subject_index)

            if epochs.shape[0] == 0:
                empty_epoch_warnings += 1
                print(f"    [INFO] {group_name} cell {cell_index+1} subject "
                      f"{subject_index+1}: 0 usable epochs.")
            else:
                print(f"    {group_name} cell {cell_index+1} subject "
                      f"{subject_index+1}: epochs shape {epochs.shape}")

            for band_idx, (band_name, (low, high)) in enumerate(BANDS.items()):
                band_values = compute_band_connectivity_epoched(epochs, low, high)

                col_idx = cell_index * N_BANDS + band_idx
                for measure_name, value in band_values.items():
                    fused_matrices[measure_name][subject_index, col_idx] = value

    if empty_epoch_warnings > 0:
        print(f"  WARNING: {empty_epoch_warnings} subject/cell combinations had 0 "
              f"epochs -> NaN written to those feature columns.")

    # per-group shape check
    for measure in MEASURE_NAMES:
        actual_shape = fused_matrices[measure].shape
        expected_shape = (n_subjects, EXPECTED_N_FEATURES)
        if actual_shape != expected_shape:
            raise ValueError(
                f"{group_name}/{measure}: expected shape {expected_shape}, "
                f"got {actual_shape}"
            )

    return fused_matrices, n_subjects


FC_feats,    n_FC    = extract_features_for_group(FC_data, "FC")
MC_feats,    n_MC    = extract_features_for_group(MC_data, "MC")
FADHD_feats, n_FADHD = extract_features_for_group(FADHD_data, "FADHD")
MADHD_feats, n_MADHD = extract_features_for_group(MADHD_data, "MADHD")


# COMBINE ACROSS GROUPS

final_features = {}
for measure in MEASURE_NAMES:
    final_features[measure] = np.vstack([
        FC_feats[measure],
        MC_feats[measure],
        FADHD_feats[measure],
        MADHD_feats[measure],
    ])

labels_list = []
labels_list += [GROUP_LABELS["FC"]]    * n_FC
labels_list += [GROUP_LABELS["MC"]]    * n_MC
labels_list += [GROUP_LABELS["FADHD"]] * n_FADHD
labels_list += [GROUP_LABELS["MADHD"]] * n_MADHD

labels = np.array(labels_list)
total_subjects = n_FC + n_MC + n_FADHD + n_MADHD

print("\nFinal feature matrix shapes (paper method, 5 bands fused, epoch-averaged):")
for measure in MEASURE_NAMES:
    actual_shape = final_features[measure].shape
    expected_shape = (total_subjects, EXPECTED_N_FEATURES)
    n_nan = int(np.isnan(final_features[measure]).sum())
    status = "OK" if actual_shape == expected_shape else "MISMATCH!"
    print(f"  {measure}: {actual_shape}  (expected {expected_shape}) [{status}] "
          f"| NaN values: {n_nan}")
    if actual_shape != expected_shape:
        raise ValueError(
            f"Final matrix for {measure} has shape {actual_shape}, "
            f"expected {expected_shape}"
        )

print(f"\nTotal Subjects: {len(labels)}, Control={np.sum(labels==0)}, ADHD={np.sum(labels==1)}")
assert len(labels) == total_subjects, "Label count does not match subject count!"

# COLUMN NAMES


column_names = []
for cell_num in range(1, N_CELLS + 1):
    ch1, ch2 = CELL_CHANNELS[cell_num]
    for band_name in BANDS.keys():
        column_names.append(f"cell{cell_num}_{ch1}-{ch2}_{band_name}")

assert len(column_names) == EXPECTED_N_FEATURES, (
    f"Expected {EXPECTED_N_FEATURES} column names, got {len(column_names)}"
)


for measure in MEASURE_NAMES:
    save_path = os.path.join(OUTPUT_DIR, f"{measure}_features_paper.npy")
    np.save(save_path, final_features[measure])

np.save(os.path.join(OUTPUT_DIR, "labels_paper.npy"), labels)

with open(os.path.join(OUTPUT_DIR, "column_names_paper.txt"), "w") as f:
    f.write("\n".join(column_names))

print(f"\nDone. Saved 4 fused feature matrices (Correlation, Coherence, PLV, PLI) to: {OUTPUT_DIR}")
print("NOTE: 5 bands used (alpha split into lower/higher), matching the paper's "
      "frequency-band structure. Connectivity for each cell/band was computed "
      "per 3-second epoch and averaged across epochs, matching the paper's "
      "epoch-based spectral estimation approach (rather than filtering the "
      "full continuous signal once).")

if np.isnan(np.vstack(list(final_features.values()))).any():
    print("\nNOTE: Some feature values are NaN because at least one subject/cell "
          "produced 0 epochs (signal shorter than 3 seconds). Handle these "
          "(e.g. impute or drop) before classification.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[VALIDATION] Group: FC
  n_cells detected: 8
  Cell 1: 13 subjects, container shape (1, 13)
  Cell 2: 13 subjects, container shape (1, 13)
  Cell 3: 13 subjects, container shape (1, 13)
  Cell 4: 13 subjects, container shape (1, 13)
  Cell 5: 13 subjects, container shape (1, 13)
  Cell 6: 13 subjects, container shape (1, 13)
  Cell 7: 13 subjects, container shape (1, 13)
  Cell 8: 13 subjects, container shape (1, 13)
  -> Validated n_subjects for FC: 13
  Sample subject0/cell0 squeezed shape: (10, 768, 2) (expect (n_epochs, 768, 2) or (768, 2) for 1 epoch)
Processing group: FC (13 subjects)
    FC cell 1 subject 1: epochs shape (10, 768, 2)
    FC cell 1 subject 2: epochs shape (10, 768, 2)
    FC cell 1 subject 3: epochs shape (10, 768, 2)
    FC cell 1 subject 4: epochs shape (10, 768, 2)
    FC cell 1 subject 5: epochs shape (10, 768, 2)
    FC cell 1 sub